# Predicting Returns

_**Predicting Returns based on specific features using Amazon Sagemaker**_
---

---
## Background
This notebook details how I've approached the assigned problem for PredictNow.ai. This solution uses AWS SageMaker for training and deployment. The deployed model can be accessed via the REST API:

https://acov36rkvg.execute-api.us-east-1.amazonaws.com/test/predictreturns

This link can be accessed using Postman, and POSTing a JSON object (examples below) with the features from the sample dataset. Please note that the columns Time and Returns should not be included as Time was not used in building the model, and Returns is the target variable which will be returned.

{"data":"-3.299642252,-3.33953055,-3.088333159,-0.211309094,0.001481741,0.596731835,0.033591572,0.0038,0.0206,0.007,0.0168,0.0054,-0.0051,0.010810811"}
{"data":"-3.041609091,-3.027403245,-2.662340394,-0.379268697,0.000924463,0.735341967,0.016453897,0.0038,0.024,0.0127,0.0202,0.0077,-0.0051,0"}

And with that, let's see how the model is deployed.


---
## Setup

After instantiating a notebook in SageMaker using the AWS Console, it is used to preprocess the data needed to train the machine learning model and to upload the data to AWS S3 storage

This cell imports the libraries required and defines environment variables required for cloud deployment

In [54]:
import boto3, re, sys, math, json, os, sagemaker, urllib.request
from sagemaker import get_execution_role
import numpy as np
import pandas as pd
from time import gmtime, strftime
from sagemaker.predictor import csv_serializer

# Define IAM role
role = get_execution_role()
prefix = 'sagemaker/PNow-assignment'

sess = sagemaker.Session()

my_region = boto3.session.Session().region_name # set the region of the instance

# this line automatically looks for the Linear Learner image URI and builds a Linear Learner container.
ll_container = sagemaker.image_uris.retrieve("linear-learner", my_region, "latest")

print("Success - the MySageMakerInstance is in the " + my_region + " region. This will use the " + ll_container + " container for your SageMaker endpoint.")

Defaulting to the only supported framework/algorithm version: 1. Ignoring framework/algorithm version: latest.


Success - the MySageMakerInstance is in the us-east-1 region. You will use the 382416733822.dkr.ecr.us-east-1.amazonaws.com/linear-learner:1 container for your SageMaker endpoint.


---
The cell below creates an S3 bucket for storing the data

---

In [55]:
bucket_name = 'pnow-assignment'
s3 = boto3.resource('s3')
try:
    if  my_region == 'us-east-1':
      s3.create_bucket(Bucket=bucket_name)
    else: 
      s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={ 'LocationConstraint': my_region })
    print('S3 bucket created successfully')
except Exception as e:
    print('S3 error: ',e)

S3 bucket created successfully


---
The cell below loads the Excel file to a dataframe. The Time variable is dropped here as it is not used for model training in this example

---

In [56]:
model_data = pd.read_excel('SPX_train_0.xlsx', engine='openpyxl')
model_data = model_data.drop(['Time'], axis=1)

---
In this cell the data is shuffled and split for training and testing. The training data is 70% of entries and is used to find the best parameters using gradient-based optimization which decreases error by minimizing the loss function. The remaining 30% is test data used to evaluate model performance on unseen data.

---

In [58]:
train_data, test_data = np.split(model_data.sample(frac=1, random_state=1729), [int(0.7 * len(model_data))])
print(train_data.shape, test_data.shape)

(630, 15) (270, 15)


---
The step below ensures the data is correctly formatted for the Amazon SageMaker pre-built Linear Learner algorithm, saves the data to the S3 bucket, and loads the data for training.

---

In [60]:
pd.concat([train_data['Returns'], train_data.drop(['Returns'], axis=1)], axis=1).to_csv('train.csv', index=False, header=False)
boto3.Session().resource('s3').Bucket(bucket_name).Object(os.path.join(prefix, 'train/train.csv')).upload_file('train.csv')
s3_input_train = sagemaker.inputs.TrainingInput(s3_data='s3://{}/{}/train'.format(bucket_name, prefix), content_type='text/csv')

---
Below the SageMaker session allows us to create an instance of the Linear Learner algorithm and define hyperparameters. These hyperparameters can also be tuned iteratively to improve metrics like increased accuracy.

---

In [62]:
sess = sagemaker.Session()
ll = sagemaker.estimator.Estimator(ll_container,role, instance_count=1, instance_type='ml.m4.xlarge',output_path='s3://{}/{}/output'.format(bucket_name, prefix),sagemaker_session=sess)
ll.set_hyperparameters(predictor_type="binary_classifier", mini_batch_size=100)

---
## Training

The code below trains the model on the data with the configurations defined above.

---

In [63]:
ll.fit({'train': s3_input_train})

2021-10-21 22:06:50 Starting - Starting the training job...
2021-10-21 22:07:05 Starting - Launching requested ML instancesProfilerReport-1634854010: InProgress
.........
2021-10-21 22:08:47 Starting - Preparing the instances for training............
2021-10-21 22:10:48 Downloading - Downloading input data...
2021-10-21 22:11:08 Training - Downloading the training image.Docker entrypoint called with argument(s): train
Running default environment configuration script
[10/21/2021 22:11:28 INFO 140222597416768] Reading default configuration from /opt/amazon/lib/python3.7/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimi

---
## Deployment

This cell deploys the model on the cloud and creates a SageMaker endpoint which will be accessed during inference

---

In [64]:
ll_predictor = ll.deploy(initial_instance_count=1,instance_type='ml.m4.xlarge')

-------------!

---
The cells below can be used to delete the created endpoint and other resources which incur costs.

---

In [69]:
# ll_predictor.delete_endpoint(delete_endpoint_config=True)

In [70]:
# bucket_to_delete = boto3.resource('s3').Bucket(bucket_name)
# bucket_to_delete.objects.all().delete()

## Conclusion

Now we have our model deployed for 'real world use'. 

As mentioned above the deployed model can be can be accessed via the REST API at:

https://acov36rkvg.execute-api.us-east-1.amazonaws.com/test/predictreturns

This link can be accessed using Postman, and POSTing a JSON object (examples below) with the features from the sample dataset. Please note the columns Time and Returns should not be included as Time was not used in building the model, and Returns is the target variable which will be returned.

{"data":"-3.299642252,-3.33953055,-3.088333159,-0.211309094,0.001481741,0.596731835,0.033591572,0.0038,0.0206,0.007,0.0168,0.0054,-0.0051,0.010810811"}
{"data":"-3.041609091,-3.027403245,-2.662340394,-0.379268697,0.000924463,0.735341967,0.016453897,0.0038,0.024,0.0127,0.0202,0.0077,-0.0051,0"}

While this implementation is not fully comprehensive I believe it does serve the purpose of providing a practical model that can be accessed, used, and iterated upon.

## Extensions

There are multiple aspects to add and iterate which would be highly valuable, especially in a production environment.

### Testing
This is an essential aspect of practical ML and while accuracy is not a consideration in this case, this implementation would be incomplete without it. I hope to discuss in more detail the relevant metrics like AUC (Area-Under-the-Curve).

### Containerization
This cloud implementation does not demonstrate the use of Docker, but for many experiments it is essential.

### Pipeline
While the cloud implementation does provide valuable dashboards and pipelines, they are not all detailed in this assignment. Again, these are valuable tools in production ML and capabilities like triggers, schedules, scaling, and monitoring enable real-time understanding and iteration for engineers.


## Summary

In this notebook this cloud offering is used to prepare, train, and deply a machine learning model. I hope this solution is viable and look forward to discussing it's strengths and shortcomings, to be able to continually improve and create value.